# 📱 Blog-to-Social-Post Converter

## What We're Building

A content repurposing tool that takes **one blog article** and generates:
1. **Twitter/X posts** — Thread of 3-5 tweets (280 chars each)
2. **LinkedIn post** — Professional, story-driven (1000-1300 chars)
3. **Twitter thread** — Detailed breakdown as a thread
4. **Email newsletter snippet** — Short teaser for email subscribers

## Why This Is Useful
Content creators spend hours adapting one article for different platforms.
Each platform has different:
- **Character limits** (Twitter: 280, LinkedIn: 3000)
- **Tone** (Twitter: punchy, LinkedIn: professional, Email: warm)
- **Formatting** (hashtags, emojis, line breaks differ by platform)

## Key Concept: Same Input, Different Prompts

We use the **same article** but pass it through **platform-specific prompts**
that encode the rules, conventions, and style of each platform.

## Stack
- **LangChain** — prompt templates and chains
- **Ollama (qwen3.5:9b)** — local LLM

## Step 1 — Install & Import Dependencies

In [ ]:
# Install if needed (uncomment)
# !pip install langchain langchain-ollama -q

from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("All imports successful!")

## Step 2 — Sample Blog Article

Here's a sample tech article our tool will repurpose.

In [ ]:
sample_blog = """
Title: Why Every Developer Should Learn Prompt Engineering in 2025

The rise of large language models has changed software development forever. 
But most developers are still treating AI like a black box — typing vague 
instructions and hoping for the best. That's like writing code without 
understanding data structures.

Prompt engineering is the skill of crafting effective instructions for AI 
models. It's not just about being polite to ChatGPT. It's a systematic 
approach that includes techniques like few-shot prompting, chain-of-thought 
reasoning, and retrieval-augmented generation.

Here's why it matters:

1. **10x Productivity Boost**: Developers who master prompt engineering 
report completing tasks 5-10x faster. Code generation, debugging, 
documentation — all accelerated.

2. **Better AI Outputs**: The difference between a vague prompt and a 
well-engineered one is like the difference between a Google search and 
a SQL query. Precision matters.

3. **New Career Opportunities**: Companies are hiring prompt engineers 
at $150-300K salaries. The role sits at the intersection of engineering, 
writing, and domain expertise.

4. **AI-Native Development**: The next generation of software won't just 
use AI as a tool — it will be built with AI at the core. Understanding 
how to communicate with models is fundamental.

The best part? You don't need a PhD in machine learning. Start with 
OpenAI's prompt engineering guide, practice with real projects, and 
experiment with different techniques. The learning curve is surprisingly 
gentle for developers who already think in structured, logical terms.

The developers who ignore this shift will find themselves increasingly 
outpaced by those who embrace it. The question isn't whether you'll need 
prompt engineering — it's whether you'll learn it now or scramble to 
catch up later.
"""

print(f"Blog article: {len(sample_blog)} chars, {len(sample_blog.split())} words")

## Step 3 — Initialize LLM

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0.6)  # Slightly creative for social content
print("LLM ready!")

## Step 4 — Twitter/X Posts Generator

Rules for Twitter/X:
- 280 characters max per tweet
- Hook in the first tweet
- Hashtags (2-3 max)
- Punchy, direct language
- Emojis are welcome

In [ ]:
twitter_prompt = ChatPromptTemplate.from_template(
    """You are a social media expert who creates viral Twitter/X content.

Convert this blog article into 3-5 standalone tweets.

RULES:
- Each tweet MUST be under 280 characters (this is critical!)
- First tweet must be a hook that makes people stop scrolling
- Use punchy, direct language
- Include 2-3 relevant hashtags in the last tweet only
- Use emojis sparingly (1-2 per tweet max)
- Each tweet should standalone (people might only see one)

Format each tweet as:
TWEET 1:
[tweet text]

TWEET 2:
[tweet text]

etc.

Blog article:
{article}

Tweets:"""
)

twitter_chain = twitter_prompt | llm | StrOutputParser()

print("🐦 Generating Twitter posts...\n")
tweets = twitter_chain.invoke({"article": sample_blog})
print(tweets)

## Step 5 — LinkedIn Post Generator

Rules for LinkedIn:
- Professional but conversational tone
- Start with a hook (first 2 lines visible before "see more")
- Line breaks for readability
- Story-driven format
- 1000-1300 characters ideal
- End with a question to drive engagement

In [ ]:
linkedin_prompt = ChatPromptTemplate.from_template(
    """You are a LinkedIn content strategist. Convert this blog article into a LinkedIn post.

RULES:
- Professional but conversational tone
- Start with a HOOK — the first 2 lines are visible before "...see more"
- Use short paragraphs (2-3 lines max)
- Use line breaks for readability (LinkedIn loves whitespace)
- Tell a story or share a perspective, not just summarize
- 1000-1300 characters total
- End with a question to encourage comments
- Add 3-5 relevant hashtags at the very end
- No emojis in the hook, sparing use elsewhere

Blog article:
{article}

LinkedIn post:"""
)

linkedin_chain = linkedin_prompt | llm | StrOutputParser()

print("💼 Generating LinkedIn post...\n")
linkedin_post = linkedin_chain.invoke({"article": sample_blog})
print(linkedin_post)
print(f"\n[Character count: {len(linkedin_post)}]")

## Step 6 — Twitter Thread Generator

A thread is different from standalone tweets — it tells a connected story.

In [ ]:
thread_prompt = ChatPromptTemplate.from_template(
    """You are a viral Twitter/X thread writer. Convert this blog article into an 
engaging thread.

RULES:
- 6-8 tweets in the thread
- Each tweet MUST be under 280 characters
- Tweet 1: Strong hook with "🧵" emoji to signal it's a thread
- Tweets 2-7: Key points, one per tweet, building on each other
- Last tweet: Call-to-action (follow, retweet, bookmark)
- Number each tweet (1/, 2/, etc.)
- Use "→" or "↓" to show continuation

Blog article:
{article}

Thread:"""
)

thread_chain = thread_prompt | llm | StrOutputParser()

print("🧵 Generating Twitter thread...\n")
thread = thread_chain.invoke({"article": sample_blog})
print(thread)

## Step 7 — Email Newsletter Snippet

A teaser for email subscribers to click through to the full article.

In [ ]:
email_prompt = ChatPromptTemplate.from_template(
    """You are an email marketing expert. Create a newsletter snippet that teases
this blog article and makes subscribers want to click through.

RULES:
- Subject line: Catchy, under 50 characters, creates curiosity
- Preview text: 40-90 characters (visible in inbox before opening)
- Body: 3-4 sentences maximum
  - Sentence 1: Intriguing hook or surprising stat
  - Sentence 2-3: The key insight or promise
  - Sentence 4: Call-to-action ("Read the full article →")
- Warm, friendly tone (you're writing to existing subscribers)
- No hard sell, just genuine value

Format:
SUBJECT: [subject line]
PREVIEW: [preview text]

BODY:
[email body]

Blog article:
{article}

Newsletter snippet:"""
)

email_chain = email_prompt | llm | StrOutputParser()

print("📧 Generating email snippet...\n")
email_snippet = email_chain.invoke({"article": sample_blog})
print(email_snippet)

## Step 8 — Complete Repurposing Pipeline

In [ ]:
def repurpose_article(article: str) -> dict:
    """
    Convert a blog article into social media content for all platforms.
    
    Args:
        article: The full blog article text
    
    Returns:
        Dict with content for each platform
    """
    print(f"📝 Article: {len(article.split())} words")
    print(f"{'='*60}\n")
    
    results = {}
    
    # Twitter posts
    print("🐦 Generating Twitter posts...")
    results["twitter_posts"] = twitter_chain.invoke({"article": article})
    print("   ✓ Done")
    
    # LinkedIn
    print("💼 Generating LinkedIn post...")
    results["linkedin"] = linkedin_chain.invoke({"article": article})
    print("   ✓ Done")
    
    # Thread
    print("🧵 Generating Twitter thread...")
    results["twitter_thread"] = thread_chain.invoke({"article": article})
    print("   ✓ Done")
    
    # Email
    print("📧 Generating email snippet...")
    results["email"] = email_chain.invoke({"article": article})
    print("   ✓ Done")
    
    print(f"\n{'='*60}")
    print("✅ All platforms generated!")
    
    return results


# Run the complete pipeline
all_content = repurpose_article(sample_blog)

In [ ]:
# Display all generated content
for platform, content in all_content.items():
    print(f"\n{'='*60}")
    print(f"📱 {platform.upper().replace('_', ' ')}")
    print(f"{'='*60}")
    print(content)
    print()

## Step 9 — Save All Content

In [ ]:
from pathlib import Path

output = ["# Social Media Content Pack\n"]
for platform, content in all_content.items():
    output.append(f"## {platform.replace('_', ' ').title()}\n")
    output.append(content)
    output.append("\n---\n")

Path("social_content.md").write_text("\n".join(output), encoding="utf-8")
print("📄 All content saved to social_content.md")

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **Platform-specific prompts** | Encode rules (char limits, tone, format) per platform |
| **Same input, multiple outputs** | One article → 4 different content pieces |
| **Prompt constraints** | Character limits, formatting rules, tone specification |
| **Pipeline pattern** | Run the same input through multiple chains |
| **Temperature tuning** | Higher (0.6-0.7) for creative social content |

## 🔧 Customization Ideas

- **More platforms**: Reddit, Instagram captions, YouTube descriptions
- **Brand voice**: Add brand guidelines to the system prompt
- **A/B variants**: Generate 2-3 versions per platform for testing
- **Image prompts**: Generate DALL-E/Midjourney prompts for social images
- **Scheduling**: Format output for Buffer/Hootsuite import